# Predictor 2: Gammatone Envelope 8

This notebook builds the 8-band gammatone envelope predictor.

Why: a single speech envelope collapses all frequencies together, while a gammatone filterbank approximates how the auditory system separates sound into frequency channels.

## Output convention

Input:

```text
/Users/yanyuwoo/Data/bids/stimuli/*.wav
```

Output:

```text
/Users/yanyuwoo/Data/Alice Comprehension/predictors/gammatone_envelope_8/
/Users/yanyuwoo/Data/Alice Comprehension/qc/gammatone_envelope_8_manifest.csv
```

Each output `.npz` contains:

- `data`: shape `(time, 8)`, sampled at 100 Hz
- `metadata`: JSON string with source file and construction parameters

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import hilbert, butter, filtfilt, resample_poly, gammatone, lfilter

BIDS_ROOT = Path('/Users/yanyuwoo/Data/bids')
STIMULI_DIR = BIDS_ROOT / 'stimuli'

ANALYSIS_ROOT = Path('/Users/yanyuwoo/Data/Alice Comprehension')
PREDICTOR_DIR = ANALYSIS_ROOT / 'predictors' / 'gammatone_envelope_8'
QC_DIR = ANALYSIS_ROOT / 'qc'

PREDICTOR_FS = 100
ENVELOPE_LOWPASS_HZ = 30.0
N_BANDS = 8
LOW_FREQ_HZ = 80.0
HIGH_FREQ_HZ = 4000.0

PREDICTOR_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

## Helper functions

In [2]:
def read_wav_mono(path: Path):
    fs, audio = wavfile.read(path)
    audio = np.asarray(audio, dtype=np.float64)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    max_abs = np.max(np.abs(audio))
    if max_abs > 0:
        audio = audio / max_abs
    return fs, audio


def require_finite(name, x):
    if not np.isfinite(x).all():
        n_bad = int((~np.isfinite(x)).sum())
        raise ValueError(f'{name} contains {n_bad} non-finite values')
    return x


def zscore_columns(x):
    x = require_finite('z-score input', np.asarray(x, dtype=np.float64))
    sd = x.std(axis=0, keepdims=True)
    if np.any(sd == 0):
        raise ValueError('Cannot z-score a constant predictor column')
    return (x - x.mean(axis=0, keepdims=True)) / sd


def lowpass(x, fs, cutoff_hz):
    nyquist = fs / 2
    cutoff_hz = min(cutoff_hz, nyquist * 0.95)
    b, a = butter(4, cutoff_hz / nyquist, btype='low')
    return filtfilt(b, a, x, axis=0)


def resample_to_predictor_fs(x, source_fs, target_fs=PREDICTOR_FS):
    if source_fs == target_fs:
        return x
    gcd = np.gcd(int(source_fs), int(target_fs))
    up = int(target_fs // gcd)
    down = int(source_fs // gcd)
    return resample_poly(x, up, down, axis=0)


def center_frequencies():
    return np.geomspace(LOW_FREQ_HZ, HIGH_FREQ_HZ, N_BANDS)


def gammatone_envelope_8(audio, fs):
    envelopes = []
    for freq in center_frequencies():
        # FIR gammatone is numerically stable for this construction step.
        b, a = gammatone(freq, 'fir', fs=fs)
        filtered = lfilter(b, a, audio)
        envelope = np.abs(hilbert(filtered))
        envelope = lowpass(envelope, fs, ENVELOPE_LOWPASS_HZ)
        envelopes.append(require_finite(f'gammatone band {freq:.1f} Hz', envelope))

    predictor = np.column_stack(envelopes)
    predictor = resample_to_predictor_fs(predictor, fs, PREDICTOR_FS)
    return zscore_columns(predictor)


def save_predictor(path: Path, data: np.ndarray, metadata: dict):
    np.savez_compressed(
        path,
        data=data.astype(np.float32),
        metadata=json.dumps(metadata),
    )

## Build gammatone envelope predictors

Run this cell when you are ready to generate the second predictor.

In [3]:
wav_paths = sorted(STIMULI_DIR.glob('*.wav'), key=lambda p: int(p.stem))
freqs = center_frequencies()
manifest_rows = []

for wav_path in wav_paths:
    stimulus_id = int(wav_path.stem)
    fs, audio = read_wav_mono(wav_path)
    predictor = gammatone_envelope_8(audio, fs)

    out_path = PREDICTOR_DIR / f'stim-{stimulus_id:02d}_gammatone_envelope_8_fs-{PREDICTOR_FS}.npz'
    metadata = {
        'stimulus_id': stimulus_id,
        'predictor_name': 'gammatone_envelope_8',
        'source_wav': str(wav_path),
        'source_fs': fs,
        'predictor_fs': PREDICTOR_FS,
        'n_bands': N_BANDS,
        'center_frequencies_hz': freqs.tolist(),
        'envelope_lowpass_hz': ENVELOPE_LOWPASS_HZ,
        'source_duration_sec': len(audio) / fs,
        'n_samples': int(predictor.shape[0]),
        'n_features': int(predictor.shape[1]),
    }
    save_predictor(out_path, predictor, metadata)

    manifest_rows.append({
        **{k: v for k, v in metadata.items() if k != 'center_frequencies_hz'},
        'center_frequencies_hz': ';'.join(f'{f:.3f}' for f in freqs),
        'path': str(out_path),
        'predictor_duration_sec': predictor.shape[0] / PREDICTOR_FS,
        'predictor_mean_abs': float(np.mean(np.abs(predictor))),
        'predictor_std_mean': float(np.mean(predictor.std(axis=0))),
    })

manifest = pd.DataFrame(manifest_rows).sort_values('stimulus_id')
manifest_path = QC_DIR / 'gammatone_envelope_8_manifest.csv'
manifest.to_csv(manifest_path, index=False)
manifest

,stimulus_id,predictor_name,source_wav,source_fs,predictor_fs,n_bands,envelope_lowpass_hz,source_duration_sec,n_samples,n_features,center_frequencies_hz,path,predictor_duration_sec,predictor_mean_abs,predictor_std_mean
0,1,gammatone_envelope_8,/Users/yanyuwoo/Data/bids/stimuli/1.wav,44100,100,8,30.0,57.540612,5755,8,80.000;139.894;244.630;427.780;748.049;1308.09...,/Users/yanyuwoo/Data/Alice Comprehension/predi...,57.55,0.665460,1.0
1,2,gammatone_envelope_8,/Users/yanyuwoo/Data/bids/stimuli/2.wav,44100,100,8,30.0,60.845193,6085,8,80.000;139.894;244.630;427.780;748.049;1308.09...,/Users/yanyuwoo/Data/Alice Comprehension/predi...,60.85,0.670723,1.0
2,3,gammatone_envelope_8,/Users/yanyuwoo/Data/bids/stimuli/3.wav,44100,100,8,30.0,63.259433,6326,8,80.000;139.894;244.630;427.780;748.049;1308.09...,/Users/yanyuwoo/Data/Alice Comprehension/predi...,63.26,0.657637,1.0
3,4,gammatone_envelope_8,/Users/yanyuwoo/Data/bids/stimuli/4.wav,44100,100,8,30.0,69.988571,6999,8,80.000;139.894;244.630;427.780;748.049;1308.09...,/Users/yanyuwoo/Data/Alice Comprehension/predi...,69.99,0.667214,1.0
4,5,gammatone_envelope_8,/Users/yanyuwoo/Data/bids/stimuli/5.wav,44100,100,8,30.0,66.272540,6628,8,80.000;139.894;244.630;427.780;748.049;1308.09...,/Users/yanyuwoo/Data/Alice Comprehension/predi...,66.28,0.666025,1.0
5,6,gammatone_envelope_8,/Users/yanyuwoo/Data/bids/stimuli/6.wav,44100,100,8,30.0,63.777551,6378,8,80.000;139.894;244.630;427.780;748.049;1308.09...,/Users/yanyuwoo/Data/Alice Comprehension/predi...,63.78,0.678141,1.0
6,7,gammatone_envelope_8,/Users/yanyuwoo/Data/bids/stimuli/7.wav,44100,100,8,30.0,62.896848,6290,8,80.000;139.894;244.630;427.780;748.049;1308.09...,/Users/yanyuwoo/Data/Alice Comprehension/predi...,62.90,0.685946,1.0
7,8,gammatone_envelope_8,/Users/yanyuwoo/Data/bids/stimuli/8.wav,44100,100,8,30.0,57.310612,5732,8,80.000;139.894;244.630;427.780;748.049;1308.09...,/Users/yanyuwoo/Data/Alice Comprehension/predi...,57.32,0.665445,1.0
8,9,gammatone_envelope_8,/Users/yanyuwoo/Data/bids/stimuli/9.wav,44100,100,8,30.0,57.226145,5723,8,80.000;139.894;244.630;427.780;748.049;1308.09...,/Users/yanyuwoo/Data/Alice Comprehension/predi...,57.23,0.673873,1.0
9,10,gammatone_envelope_8,/Users/yanyuwoo/Data/bids/stimuli/10.wav,44100,100,8,30.0,61.269660,6127,8,80.000;139.894;244.630;427.780;748.049;1308.09...,/Users/yanyuwoo/Data/Alice Comprehension/predi...,61.27,0.663250,1.0
